In [1]:
# MLflow Plotter Demo - visualize experiment results
from pybpr.plotter import MLflowPlotter
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
# Initialize plotter with path to mlflow.db
plotter = MLflowPlotter(tracking_uri="mlflow.db")

In [3]:
# List all experiments
experiments = plotter.get_experiments()
print("Available experiments:")
experiments

Available experiments:


,experiment_id,name,artifact_location
0,1,movielens,/home/rsandhu/pybpr/examples/mlruns/1
1,0,Default,/home/rsandhu/pybpr/examples/mlruns/0


In [4]:
# Get runs for a specific experiment
exp_name = "movielens"
runs = plotter.get_runs(experiment_name=exp_name)
print(f"Runs in '{exp_name}':")
runs

Runs in 'movielens':


,run_id,run_name,status,start_time,model.n_latent,model.sparse,model.init_std,optimizer.name,optimizer.lr,optimizer.weight_decay,...,model.use_global_bias,model.dropout,model.activation,training.eval_user_size,eval_user_size,eval_auc_neg_ratio,data.train_ratio_neg,sweep.data.train_ratio_neg,auc_std,loss
0,5cb69d1a725d4b28b21ae8195719e285,ml-100k_indicator,FINISHED,1772557595676,64,False,0.1,Adam,0.001,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a615d3ba5009498d8a011c87da4d7843,ml-100k_indicator,FINISHED,1772494215541,64,False,0.1,Adam,0.001,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,d1c46a07f2bd4cd8a78193a8fc957ad0,ml-100k_indicator,FAILED,1772494048501,64,False,0.1,Adam,0.001,0.0001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,042a895e19ff4d859b172769374321b2,ml-100k_indicator,FAILED,1772488269963,64,False,NaN,Adam,0.001,0.0001,...,True,0.0,None,None,None,-1,NaN,NaN,NaN,NaN
4,a7ead96b4395448f9469b768b247acce,ml-100k_indicator,FAILED,1772481195770,64,False,NaN,Adam,0.001,0.0001,...,True,0.0,None,None,None,-1,0.0,"[0.0, 0.5, 0.8, 1.0]",NaN,NaN
5,6626a84ff66a4e3783f66bc0b9dfac78,ml-100k_indicator,FINISHED,1772480354741,64,False,NaN,Adam,0.001,0.0001,...,True,0.0,None,None,None,-1,0.0,"[0.0, 0.5, 0.8, 1.0]",0.157571,0.60053


In [7]:
run_id = '5cb69d1a725d4b28b21ae8195719e285'
histories = plotter.get_run_metrics_history(run_id=run_id)
df = None
for metric, hist in histories.items():
    hist = hist.rename(columns={"value": metric})[["step", metric]]
    df = hist if df is None else df.merge(hist, on="step", how="outer")
df = df.sort_values("step").reset_index(drop=True)
df.head(10)

,step,train_bpr_loss,auc,ndcg_10,precision_10,recall_10,bpr_loss
0,1,0.694889,NaN,NaN,NaN,NaN,NaN
1,2,0.690211,NaN,NaN,NaN,NaN,NaN
2,3,0.688182,NaN,NaN,NaN,NaN,NaN
3,4,0.685187,NaN,NaN,NaN,NaN,NaN
4,5,0.680437,NaN,NaN,NaN,NaN,NaN
5,6,0.677685,NaN,NaN,NaN,NaN,NaN
6,7,0.675793,NaN,NaN,NaN,NaN,NaN
7,8,0.674031,NaN,NaN,NaN,NaN,NaN
8,9,0.668070,NaN,NaN,NaN,NaN,NaN
9,10,0.667889,0.538469,0.275395,0.236344,0.159684,0.689766


In [ ]:
# Create summary table with final metrics
summary = plotter.summary_table(
    experiment_name=exp_name,
    #metrics=["test_auc", "train_loss"],
    params=["n_latent", "lr"]
)
print("Run Summary (sorted by test_auc):")
summary

In [ ]:

fig = plotter.plot_single_run(
    run_id='81642b1cf43a44fd9ca9b796ed20c374',
    figsize=(14, 5),
    std_width=2.0,
    show_std=True,
)
plt.show()

In [ ]:
# Compare all runs with train_loss and test_auc side by side
fig = plotter.plot_runs_comparison(
    experiment_name=exp_name,
    metrics=["train_loss", "test_auc"],
    figsize=(14, 5),
    std_width=1.0,
    show_std=True
)
plt.show()

In [ ]:
# Compare runs with different std_width
fig = plotter.plot_runs_comparison(
    experiment_name=exp_name,
    metrics=["train_loss", "test_auc"],
    figsize=(14, 5),
    std_width=2.0,
    show_std=True
)
plt.show()

In [ ]:
# Plot top 3 runs by test_auc with std bands
fig = plotter.plot_best_runs(
    experiment_name=exp_name,
    metric="test_auc",
    n_best=3,
    plot_metrics=["train_loss", "test_auc"],
    figsize=(14, 5),
    std_width=2.0,
    show_std=True
)
plt.show()

In [ ]:
# Get metric history for custom analysis
if len(runs) > 0:
    run_id = runs.iloc[0]["run_id"]
    histories = plotter.get_run_metrics_history(
        run_id=run_id,
        metric_keys=["train_loss", "test_auc"]
    )
    print("Train Loss history:")
    print(histories["train_loss"].head())
    print("\nTest AUC history:")
    print(histories["test_auc"].head())

In [ ]:
# Analyze grid search experiment results
grid_exp_name = "example4_grid_search"

try:
    grid_runs = plotter.get_runs(
        experiment_name=grid_exp_name
    )
    print(f"Grid search runs: {len(grid_runs)}")
    
    # Create summary table
    grid_summary = plotter.summary_table(
        experiment_name=grid_exp_name,
        metrics=["test_auc", "train_loss"],
        params=["n_latent", "lr", "loss_function"]
    )
    print("\nGrid search results:")
    display(grid_summary)
    
    # Plot comparison with std bands
    fig = plotter.plot_runs_comparison(
        experiment_name=grid_exp_name,
        metrics=["train_loss", "test_auc"],
        figsize=(14, 5),
        std_width=1.0,
        show_std=True
    )
    plt.show()
    
except ValueError as e:
    msg = f"Grid search experiment not found: {e}"
    print(msg)
    print("Run Example 4 from simple_pipeline_example.py")

In [ ]:
# Create and save plot to file
fig = plotter.plot_runs_comparison(
    experiment_name=exp_name,
    metrics=["train_loss", "test_auc"],
    figsize=(14, 5),
    std_width=1.5,
    show_std=True
)

# Save to file
fig.savefig(
    "experiment_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
print("Plot saved to experiment_comparison.png")
plt.show()